In [ ]:
import urllib.request
import torch
import time
import numpy as np
import psutil

print("--- END-TO-END PRION PIPELINE TEST ---")
print(f"Initial System RAM: {psutil.virtual_memory().percent}%\n")

# 1. TARGET ACQUISITION (Pulling real Prion Data from PDB)
pdb_id = "1qlx"
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
print(f"Downloading Human Prion Protein ({pdb_id.upper()})...")

response = urllib.request.urlopen(url)
pdb_data = response.read().decode('utf-8')

# Parse the 3D coordinates of every atom in the prion
prion_coords = []
for line in pdb_data.split('\n'):
    if line.startswith("ATOM"):
        # PDB format strictly defines coordinates at these character indices
        x = float(line[30:38].strip())
        y = float(line[38:46].strip())
        z = float(line[46:54].strip())
        prion_coords.append([x, y, z])

num_atoms = len(prion_coords)
print(f"Successfully parsed {num_atoms} atoms from the Prion structure.")

# 2. FOLDPIPE SERIALIZATION (Simulating a trajectory of 10,000 frames)
num_frames = 10000
print(f"\nSerializing a {num_frames}-frame trajectory into FoldPipe architecture...")

base_tensor = torch.tensor(prion_coords, dtype=torch.float32)
trajectory_tensor = base_tensor.unsqueeze(0).repeat(num_frames, 1, 1)
noise = torch.randn_like(trajectory_tensor) * 0.1 
trajectory_tensor += noise

print(f"Flat Tensor Shape: {trajectory_tensor.shape} | Memory footprint: {trajectory_tensor.element_size() * trajectory_tensor.nelement() / (1024**2):.2f} MB")

# 3. ASYNC GPU SIMULATION LOOP (The MLFF bottleneck test)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nPushing massive graph to {device.type.upper()}...")

pinned_trajectory = trajectory_tensor.pin_memory()

batch_size = 64
num_batches = num_frames // batch_size

start_sim = time.time()
if torch.cuda.is_available():
    torch.cuda.synchronize()

for i in range(num_batches):
    batch = pinned_trajectory[i*batch_size : (i+1)*batch_size]
    batch_gpu = batch.to(device, non_blocking=True)
    
    if torch.cuda.is_available():
        dummy_weights = torch.randn((batch_size, num_atoms, num_atoms), device=device)
        _ = torch.bmm(dummy_weights, batch_gpu)
        torch.cuda.synchronize()

end_sim = time.time()

print("\n" + "="*50)
print("END-TO-END PRION SIMULATION RESULTS")
print("="*50)
print(f"Total Frames Processed : {num_frames}")
print(f"Simulation Time        : {end_sim - start_sim:.2f} seconds")
print(f"Throughput             : {num_frames / (end_sim - start_sim):.2f} frames/sec")
print(f"System RAM Peak        : {psutil.virtual_memory().percent}%")
if torch.cuda.is_available():
    print(f"GPU VRAM Peak          : {torch.cuda.max_memory_allocated() / (1024 ** 2):.1f} MB")
print("="*50)
print("SUCCESS: A massive macromolecule processed without triggering Kaggle OOM.")
